# Transformer

---

## 一、Input Embeddings

In [1]:
import torch
import torch.nn as nn
import math
import torch.nn.functional as F

class InputEmbedding(nn.Module):
    """
    输入的嵌入向量表, 将单词序号转化为嵌入向量
    d_model: 嵌入向量的维度
    vocab_size: 词表大小
    """
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model)
     
    """
    直接将单词id传入embedding_table即可
    这里原文提出要额外乘以一个权重sqrt(d_model)
    """  
    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)

---

## 二、Positional Embeddings

In [2]:
class PositionalEmbedding(nn.Module):
    """
    位置嵌入向量和单词嵌入向量维度相同, 都是d_model, 便于两者逐元素相加
    这里采用了一维位置编码, sinusoidal positional encoding
    """
    def __init__(self, d_model: int, seq_len: int):
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        
        """
        构建pos_mat和i2_mat, 计算三角函数内的值
        pos_mat形状为(seq_len, 1)
        i2_mat形状为(1, model_dim/2)
            假设2*i=0,2,4,6
            则pe中的偶数列(2i)的嵌入值为2*i
            则pe中的奇数列(2i+1)也嵌入值为2*i
            另一种求法参考**position_embedding节**
        """
        pe = torch.zeros(seq_len, d_model)
        pos_mat = torch.arange(seq_len).reshape((-1, 1))
        i2_mat = torch.pow(10000, torch.arange(0, d_model, 2).reshape((1, -1)) / d_model)
        
        pe[:, 0::2] = torch.sin(pos_mat / i2_mat)
        pe[:, 1::2] = torch.cos(pos_mat / i2_mat)
        
        """
        支持批处理, 形状为(1, seq_len, d_model)
        将不可学习的张量保存在模型参数文件中, 需要将其注册为缓冲区
        """
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    """
    对输入批次x中的每个句子叠加位置嵌入
    手动修改pe的requires_grad属性为False, 表示不更新位置嵌入编码
    """
    def forward(self, x):
        # input_embedding + positional_embedding
        x = x + (self.pe[:, x.shape[1], :]).requires_grad_(False)
        return x

---

## 三、FeedForwardBlock

In [3]:
class FeedForwardBlock(nn.Module):
    """
    构造两层前向连接层FFN(x)
    FFN(x) = max(0, xW1 + b1)W2 + b2
    """
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff) # W1, b1
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(d_ff, d_model) # W2, b2
        
    """
    (batch, seq_len, d_model) -> (batch, seq_len, d_ff) -> (batch, seq_len, d_model)
    """
    def forward(self, x):
        x = self.linear1(x)
        x = self.dropout(F.relu(x))
        x = self.linear2(x)
        return x

---

## 四、MultiHead Self-Attention

In [4]:
class MultiHeadAttentionBlock(nn.Module):
    """
    Attention(Q, K, V) = softmax(QK^T / sqrt(d_k))V
    一个头负责访问批次内整个句子的一部分内容
    """
    def __init__(self, d_model: int, num_heads: int, dropout: float):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_k = d_model // num_heads
        self.w_q = nn.Linear(d_model, d_model) # Wq
        self.w_k = nn.Linear(d_model, d_model) # Wk
        self.w_v = nn.Linear(d_model, d_model) # Wv
        
        self.w_o = nn.Linear(d_model, d_model) # Wo
        self.dropout = nn.Dropout(dropout)
        
    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):
        d_k = query.shape[-1]
        
        """
        (batch, num_heads, seq_len, d_k) @ (batch, num_heads, d_k, seq_len) -> (batch, num_heads, seq_len, seq_len)
        除以sqrt(d_k)是为了让注意力分数分布更加集中, 避免梯度消失
        """
        attention_score = (query @ key.transpose(-2, -1) / math.sqrt(d_k))
        if mask is not None:
            attention_score = attention_score.masked_fill_(mask == 0, -1e9)
        attention_score = attention_score.softmax(dim=-1)  # (batch, num_heads, seq_len, seq_len)
        if dropout is not None:
            attention_score = dropout(attention_score)
           
        """
        最后与多头的V相乘
        (batch, num_heads, seq_len, seq_len) @ (batch, num_heads, seq_len, d_k) -> (batch, num_heads, seq_len, d_k)
        """ 
        return (attention_score @ value)
         
         
    def forward(self, q, k, v, mask):
        """
        利用输入的x, 分三路同时得到Q, K, V
        (batch, seq_len, d_model) -> (batch, seq_len, d_model)
        """
        query = self.w_q(q)
        key   = self.w_k(k)
        value = self.w_v(v)
        
        """
        把Q, K, V分别拆分成多个头: (batch, seq_len, d_model) -> (batch, num_heads, seq_len, d_k)
        保证一个head能看到批次内的所有句子(的一部分), 即所有的(seq_len, d_k)
        """
        query = query.view(query.shape[0], self.num_heads, query.shape[1], self.d_k)
        key   = key.view(key.shape[0], self.num_heads, key.shape[1], self.d_k)
        value = value.view(value.shape[0], self.num_heads, value.shape[1], self.d_k)
        
        """
        将按head拆分过的Q, K, V输入到多头注意力机制
        x形状为(batch_size, num_heads, seq_len, d_k), 与多头query, key, value的形状相同
        """
        x = MultiHeadAttentionBlock.attention(query, key, value, mask, self.dropout)
        
        """
        先把多个头重新concat起来: (batch, num_heads, seq_len, d_k) -> (batch, seq_len, num_heads * d_k) = (batch, seq_len, d_model)
        再乘以矩阵Wo: (batch, seq_len, d_model) -> (batch, seq_len, d_model)
        """
        x = x.transpose(1, 2).reshape(x.shape[0], -1, self.num_heads * self.d_k)
        return self.w_o(x)

---

## 五、Residual Connection & LayerNormalization

In [5]:
class LayerNormalization(nn.Module):
    def __init__(self, eps: float=10**-6):
        super().__init__()
        self.eps = eps
        """
        gamma和beta都是可学习的参数
        由nn.Parameter决定
        """
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
        
    def forward(self, x):
        """
        层归一化求解**一个样本内部**各分量的均值和方差
        形状为 (batch, seq_len, num_features) -> (batch, seq_len, 1)
        """
        mean = x.mean(dim=-1, keepdim=True)
        std  = x.std(dim=-1, keepdim=True)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta # 加eps防止除以0

class ResidualConnection(nn.Module):
    def __init__(self, dropout: float):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = LayerNormalization()
        
    def forward(self, x, sublayer):
        """ 
        x是旁路输入, sublayer是下游直接输入
        这里按照原文描述, 先得到sublayer, 再执行layer_norm
        """
        return self.dropout(self.layer_norm(x + sublayer(x)))

---

## 六、Encoder Block 与 Encoder

In [6]:
class EncoderBlock(nn.Module):
    def __init__(self, self_attn_block: MultiHeadAttentionBlock, ffn_block: FeedForwardBlock, dropout: float):
        super().__init__()
        self.self_attn_block = self_attn_block
        self.ffn_block = ffn_block
        self.residual_connection1, self.residual_connection2 = ResidualConnection(dropout), ResidualConnection(dropout)
        
    def forward(self, x, src_mask):
        """
        由于三个输入都是x自己, 所以是**自注意力**机制
        这里利用lambda表达式表示多头自注意力下游块, 输入三次x自己
        """
        x = self.residual_connection1(x, lambda x: self.self_attn_block(x, x, x, src_mask))
        x = self.residual_connection2(x, self.ffn_block)

In [7]:
class Encoder(nn.Module):
    def __init__(self, layers: nn.ModuleList):
        super().__init__()
        self.layers = layers
        self.layer_norm = LayerNormalization()
    
    """
    按照原文经过N个Encoder Blocks
    最终输出层归一化结果给Decoder
    """
    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.layer_norm(x)

---

## 七、Decoder Block 和 Decoder

In [8]:
class DecoderBlock(nn.Module):
    def __init__(self, self_attn_block: MultiHeadAttentionBlock, cross_attn_block: MultiHeadAttentionBlock, ffn_block: FeedForwardBlock, dropout: float):
        super().__init__()
        self.self_attn_block = self_attn_block
        self.cross_attn_block = cross_attn_block
        self.ffn_block = ffn_block
        self.residual_connection1, self.residual_connection2, self.residual_connection3 = ResidualConnection(dropout), ResidualConnection(dropout), ResidualConnection(dropout)
        
    """
    第一个残差连接中的attn与Encoder中的**自注意力**机制相同, 只是掩码不同
    第二个残差连接中的attn使用了**交叉注意力**机制: query来自Decoder本身, 而key和value来自Encoder的输出
    第三个残差连接与Encoder中的相同
    """
    def forward(self, x, encoder_output, src_mask, tgt_mask):
        x = self.residual_connection1(x, lambda x: self.self_attn_block(x, x, x, tgt_mask))
        x = self.residual_connection2(x, lambda x: self.cross_attn_block(x, encoder_output, encoder_output, src_mask))
        x = self.residual_connection3(x, self.ffn_block)
        return x

In [9]:
class Decoder(nn.Module):
    def __init__(self, layers: nn.ModuleList):
        super().__init__()
        self.layers = layers
        self.layer_norm = LayerNormalization()
        
    """
    与Encoder类似, Decoder也包含多个子层
    Decoder的输入包含了Encoder的输出
    """
    def forward(self, x, encoder_output, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        return self.layer_norm(x)

---

## 八、Projection Linear Layer

In [10]:
class ProjectionLayer(nn.Module):
    """
    定义投影层, 将嵌入向量投影回词表空间
    (batch, seq_len, d_model) -> (batch, seq_len, vocab_size)
    """
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab_size)
        
    def forward(self, x):
        return F.log_softmax(self.proj(x), dim=-1)